In [ ]:
# ===================== [1] SETUP =====================
!pip install segmentation_models_pytorch albumentations timm pydensecrf --quiet

import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch.nn as nn
from pydensecrf.utils import unary_from_softmax
from pydensecrf.densecrf import DenseCRF2D

# ===================== [2] CONFIG =====================
IMG_DIR = "./images"
MASK_DIR = "./masks"
IMG_SIZE = 512
BATCH_SIZE = 4
EPOCHS = 60
SAVE_FREQ = 5
MODEL_PATH = "./best_model_ensemble.pth"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SCALER = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
print("Device:", DEVICE)

# ===================== [3] DATA =====================
class PolypDataset(Dataset):
    def __init__(self, img_dir, mask_dir, files, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.files[idx])
        mask_path = os.path.join(self.mask_dir, self.files[idx])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype('float32')

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"].unsqueeze(0)

        return image, mask

files = sorted(os.listdir(IMG_DIR))
split = int(0.8 * len(files))
train_files, val_files = files[:split], files[split:]

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.5),
    A.GaussNoise(p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.CoarseDropout(p=0.3),
    A.Normalize(),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
])

train_dataset = PolypDataset(IMG_DIR, MASK_DIR, train_files, train_transform)
val_dataset = PolypDataset(IMG_DIR, MASK_DIR, val_files, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ===================== [4] MODELS =====================
model1 = smp.UnetPlusPlus(encoder_name="efficientnet-b4", encoder_weights="imagenet", in_channels=3, classes=1).to(DEVICE)
model2 = smp.DeepLabV3Plus(encoder_name="resnet50", encoder_weights="imagenet", in_channels=3, classes=1).to(DEVICE)
model3 = smp.FPN(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(DEVICE)

models = [model1, model2, model3]

# ===================== [5] LOSSES =====================
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.5, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        TP = (inputs * targets).sum()
        FP = ((1 - targets) * inputs).sum()
        FN = (targets * (1 - inputs)).sum()
        return 1 - (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)

class FocalTverskyLoss(nn.Module):
    def __init__(self, gamma=0.75):
        super().__init__()
        self.tversky = TverskyLoss()
        self.gamma = gamma

    def forward(self, inputs, targets):
        tversky_loss = self.tversky(inputs, targets)
        return torch.pow(tversky_loss, self.gamma)

bce = nn.BCEWithLogitsLoss()
tversky = TverskyLoss()
focal = FocalTverskyLoss()

def loss_fn(pred, target):
    return 0.3 * bce(pred, target) + 0.3 * tversky(pred, target) + 0.4 * focal(pred, target)

def dice_score(preds, targets, threshold=0.5):
    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()
    intersection = (preds * targets).sum()
    return (2. * intersection) / (preds.sum() + targets.sum() + 1e-8)

# ===================== [6] TRAIN =====================
optimizers = [torch.optim.AdamW(m.parameters(), lr=1e-4) for m in models]
best_dice = 0

for epoch in range(EPOCHS):
    [m.train() for m in models]
    train_loss = 0

    for images, masks in tqdm(train_loader, desc=f"[Train] Epoch {epoch+1}"):
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        for opt in optimizers:
            opt.zero_grad()

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            preds = [m(images) for m in models]
            ensemble_pred = torch.mean(torch.stack(preds), dim=0)
            loss = loss_fn(ensemble_pred, masks)

        SCALER.scale(loss).backward()
        for opt in optimizers:
            SCALER.step(opt)
        SCALER.update()
        train_loss += loss.item()

    # ===== Validation =====
    [m.eval() for m in models]
    val_loss, val_dice = 0, 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                preds = [m(images) for m in models]
                ensemble_pred = torch.mean(torch.stack(preds), dim=0)
                val_loss += loss_fn(ensemble_pred, masks).item()
                val_dice += dice_score(ensemble_pred, masks).item()

    val_loss /= len(val_loader)
    val_dice /= len(val_loader)
    print(f"[Epoch {epoch+1}] Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f} | Dice: {val_dice:.4f}")

    # Save best
    if val_dice > best_dice:
        best_dice = val_dice
        for i, m in enumerate(models):
            torch.save(m.state_dict(), f"./best_model_{i+1}.pth")
        print("✅ Best model saved.")

# ===================== [7] INFERENCE =====================
for i in range(3):
    img, mask = val_dataset[i]
    with torch.no_grad():
        img_tensor = img.unsqueeze(0).to(DEVICE)
        preds = [m(img_tensor) for m in models]
        pred = torch.mean(torch.stack(preds), dim=0)
        pred = torch.sigmoid(pred).squeeze().cpu().numpy()
        pred = (pred > 0.5).astype(np.uint8)

    # Optional CRF
    raw_img = img.permute(1, 2, 0).cpu().numpy()
    crf_result = apply_crf(raw_img, pred)

    fig, ax = plt.subplots(1, 4, figsize=(15, 5))
    ax[0].imshow(raw_img)
    ax[0].set_title("Image")
    ax[1].imshow(mask.squeeze(), cmap='gray')
    ax[1].set_title("Ground Truth")
    ax[2].imshow(pred, cmap='gray')
    ax[2].set_title("Prediction")
    ax[3].imshow(crf_result, cmap='gray')
    ax[3].set_title("CRF Result")
    plt.show()
